# Partea 1 – Opțiunea B: Regresie pentru Estimarea Prețurilor Laptopurilor
**Dataset:** `laptop Price Prediction Dataset.csv`  
**Target:** `Price ($)` (valoare numerică continuă)  
**Scop:** Antrenăm un model de regresie care estimează prețul unui laptop pe baza specificațiilor tehnice.


## 1. Import librării

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
os.makedirs("photos", exist_ok=True)
print("Librării încărcate cu succes.")

## 2. Încărcare date

In [2]:
df = pd.read_csv("laptop Price Prediction Dataset.csv")
print(f"Dimensiune dataset: {df.shape[0]} rânduri × {df.shape[1]} coloane")
df.head()


Dimensiune dataset: 3000 rânduri × 12 coloane


,Brand,Model,Processor,RAM (GB),Storage (GB),Screen Size (inches),Graphics Card,Operating System,Weight (kg),Battery Life (hours),Price ($),Warranty (years)
0,MSI,Model 0,Intel i7,16,512,13.3,Intel UHD,Windows 10,1.74,3.9,1816.03,1
1,Lenovo,Model 1,AMD Ryzen 9,16,256,13.3,Intel UHD,Windows 10,2.40,14.4,2452.06,3
2,Asus,Model 2,AMD Ryzen 5,32,2048,17.0,AMD Radeon,Linux,1.99,14.0,2422.28,3
3,MSI,Model 3,AMD Ryzen 7,64,2048,17.0,Intel UHD,Linux,2.69,11.2,1385.55,2
4,Apple,Model 4,AMD Ryzen 7,64,2048,14.0,NVIDIA GTX 1650,Linux,1.14,8.6,2235.69,2


## 3. Explorare date (EDA) și verificare echilibru
Verificăm:
- valori lipsă
- distribuția variabilei țintă `Price ($)`
- echilibrul / dispersia fiecărei coloane


In [3]:
print("=== Tipuri de date ===")
print(df.dtypes)
print()
print("=== Valori lipsă ===")
print(df.isnull().sum())


=== Tipuri de date ===
Brand                       str
Model                       str
Processor                   str
RAM (GB)                  int64
Storage (GB)              int64
Screen Size (inches)    float64
Graphics Card               str
Operating System            str
Weight (kg)             float64
Battery Life (hours)    float64
Price ($)               float64
Warranty (years)          int64
dtype: object

=== Valori lipsă ===
Brand                   0
Model                   0
Processor               0
RAM (GB)                0
Storage (GB)            0
Screen Size (inches)    0
Graphics Card           0
Operating System        0
Weight (kg)             0
Battery Life (hours)    0
Price ($)               0
Warranty (years)        0
dtype: int64


In [4]:
print("=== Statistici descriptive ===")
df.describe(include="all").T


=== Statistici descriptive ===


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Brand,3000,8,Dell,400,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Model,3000,3000,Model 0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Processor,3000,6,AMD Ryzen 7,516,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RAM (GB),3000.0,NaN,NaN,NaN,30.189333,21.395811,8.0,16.0,32.0,64.0,64.0
Storage (GB),3000.0,NaN,NaN,NaN,953.941333,682.412093,256.0,512.0,512.0,1024.0,2048.0
Screen Size (inches),3000.0,NaN,NaN,NaN,14.9203,1.436644,13.3,13.3,14.0,15.6,17.0
Graphics Card,3000,5,AMD Radeon,634,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Operating System,3000,4,Linux,759,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Weight (kg),3000.0,NaN,NaN,NaN,2.247293,0.729991,1.0,1.6075,2.23,2.9,3.5
Battery Life (hours),3000.0,NaN,NaN,NaN,9.039433,3.445196,3.0,6.1,9.1,12.0,15.0


In [5]:
# ── Distribuția prețului (target)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["Price ($)"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Distribuția Price ($)")
axes[0].set_xlabel("Preț ($)")
axes[0].set_ylabel("Frecvență")

import scipy.stats as stats
stats.probplot(df["Price ($)"], plot=axes[1])
axes[1].set_title("Q-Q Plot Price ($)")

plt.tight_layout()
plt.savefig("photos/price_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Skewness: {df['Price ($)'].skew():.4f}  |  Kurtosis: {df['Price ($)'].kurtosis():.4f}")

Skewness: -0.0388  |  Kurtosis: -1.1825


In [6]:
# ── Preț mediu per categorie
cat_cols = ["Brand", "Processor", "Graphics Card", "Operating System"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, col in zip(axes.flat, cat_cols):
    order = df.groupby(col)["Price ($)"].mean().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y="Price ($)", order=order, ax=ax, palette="Set2")
    ax.set_title(f"Price ($) per {col}")
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("photos/price_per_category.png", dpi=120, bbox_inches="tight")
plt.show()

In [7]:
# ── Corelații numerice
num_df = df.select_dtypes(include="number")
corr = num_df.corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Matricea de corelație (variabile numerice)")
plt.tight_layout()
plt.savefig("photos/correlation_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nCorelație cu Price ($):")
print(num_df.corr()["Price ($)"].sort_values(ascending=False))


Corelație cu Price ($):
Price ($)               1.000000
RAM (GB)                0.013503
Storage (GB)            0.003536
Warranty (years)       -0.003077
Weight (kg)            -0.008928
Battery Life (hours)   -0.009235
Screen Size (inches)   -0.013560
Name: Price ($), dtype: float64


In [8]:
# ── Frecvența valorilor categoriale (echilibru clase)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flat, cat_cols):
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values, color=sns.color_palette("Set2", len(counts)))
    ax.set_title(f"Distribuție {col}")
    ax.set_ylabel("Nr. înregistrări")
    ax.tick_params(axis="x", rotation=15)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(val), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("photos/category_balance.png", dpi=120, bbox_inches="tight")
plt.show()

In [9]:
# ── Echilibrul valorilor numerice (RAM, Storage)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ram_counts = df["RAM (GB)"].value_counts().sort_index()
storage_counts = df["Storage (GB)"].value_counts().sort_index()

axes[0].bar(ram_counts.index.astype(str), ram_counts.values, color="cornflowerblue", edgecolor="white")
axes[0].set_title("Distribuție RAM (GB)")
axes[0].set_xlabel("RAM (GB)")
axes[0].set_ylabel("Nr. înregistrări")

axes[1].bar(storage_counts.index.astype(str), storage_counts.values, color="coral", edgecolor="white")
axes[1].set_title("Distribuție Storage (GB)")
axes[1].set_xlabel("Storage (GB)")
axes[1].set_ylabel("Nr. înregistrări")

plt.tight_layout()
plt.savefig("photos/ram_storage_balance.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Preprocesare
Pași:
1. **Eliminăm** coloana `Model` (e un index fără valoare predictivă: „Model 0", „Model 1" etc.)
2. **Encoding ordinal** pentru `Graphics Card` — există o ordine naturală a puterii GPU
3. **One-Hot Encoding** pentru `Brand`, `Processor`, `Operating System`
4. **StandardScaler** pentru variabilele numerice


In [10]:
df = df.drop(columns=["Model"])

GPU_ORDER = [["Intel UHD", "AMD Radeon", "NVIDIA GTX 1650", "NVIDIA RTX 3060", "NVIDIA RTX 3070"]]

cat_ohe = ["Brand", "Processor", "Operating System"]
cat_ord = ["Graphics Card"]
num_cols = ["RAM (GB)", "Storage (GB)", "Screen Size (inches)",
            "Weight (kg)", "Battery Life (hours)", "Warranty (years)"]
target = "Price ($)"

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Target range: {y.min():.2f} – {y.max():.2f}  |  Media: {y.mean():.2f}  |  Std: {y.std():.2f}")


Train: 2400 | Test: 600
Target range: 501.53 – 2999.68  |  Media: 1780.33  |  Std: 719.90


In [11]:
preprocessor = ColumnTransformer([
    ("num",     StandardScaler(),                                             num_cols),
    ("cat_ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False),  cat_ohe),
    ("cat_ord", OrdinalEncoder(categories=GPU_ORDER),                         cat_ord),
])
print("Preprocessor definit.")


Preprocessor definit.


## 5. Antrenare modele
Comparăm 3 modele:
- **Ridge Regression** — model liniar regularizat
- **Random Forest Regressor** — ansamblu de arbori (bagging)
- **Gradient Boosting Regressor** — ansamblu secvențial (boosting)


In [12]:
models = {
    "Ridge Regression":    Ridge(alpha=1.0),
    "Random Forest":       RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                      max_depth=5, random_state=42),
}

results = {}
trained_pipes = {}

for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    
    results[name] = {"MAE": mae, "RMSE": rmse, "R²": r2}
    trained_pipes[name] = pipe
    print(f"{name:25s}  →  MAE={mae:7.2f}  RMSE={rmse:7.2f}  R²={r2:.4f}")


Ridge Regression           →  MAE= 638.47  RMSE= 734.72  R²=-0.0120
Random Forest              →  MAE= 645.24  RMSE= 747.80  R²=-0.0484
Gradient Boosting          →  MAE= 658.31  RMSE= 771.39  R²=-0.1156


## 6. Comparare modele și selectarea celui mai bun

In [13]:
results_df = pd.DataFrame(results).T.sort_values("R²", ascending=False)
print(results_df.to_string())

best_name = results_df.index[0]
best_pipe  = trained_pipes[best_name]
print(f"\n✔ Cel mai bun model: {best_name}")


                          MAE        RMSE        R²
Ridge Regression   638.470780  734.718289 -0.012006
Random Forest      645.237293  747.803534 -0.048374
Gradient Boosting  658.307549  771.391145 -0.115554

✔ Cel mai bun model: Ridge Regression


In [14]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = ["MAE", "RMSE", "R²"]
colors  = ["#4C72B0", "#DD8452", "#55A868"]

for ax, metric, color in zip(axes, metrics, colors):
    vals = [results[n][metric] for n in results]
    bars = ax.bar(list(results.keys()), vals, color=color, edgecolor="white")
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=15)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001 * max(vals),
                f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Comparare metrici modele", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("photos/model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 7. Cross-Validation (5-fold) pe cel mai bun model

In [15]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2   = cross_val_score(best_pipe, X, y, cv=kf, scoring="r2",                n_jobs=-1)
cv_rmse = cross_val_score(best_pipe, X, y, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1)

print(f"Model: {best_name}")
print(f"CV R²   per fold: {cv_r2.round(4)}  |  Media: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"CV RMSE per fold: {(-cv_rmse).round(2)}  |  Media: {(-cv_rmse).mean():.2f} ± {(-cv_rmse).std():.2f}")


Model: Ridge Regression
CV R²   per fold: [-0.012  -0.0105 -0.0131 -0.0217 -0.0144]  |  Media: -0.0144 ± 0.0039
CV RMSE per fold: [734.72 717.56 728.14 726.78 715.49]  |  Media: 724.54 ± 7.10


## 8. Vizualizări – Predicted vs Actual & Reziduuri

In [16]:
y_pred_best = best_pipe.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred_best, alpha=0.4, edgecolors="k", linewidths=0.2, color="steelblue")
lims = [min(y_test.min(), y_pred_best.min()) - 50, max(y_test.max(), y_pred_best.max()) + 50]
axes[0].plot(lims, lims, "r--", lw=1.5)
axes[0].set_xlabel("Preț real ($)")
axes[0].set_ylabel("Preț estimat ($)")
axes[0].set_title(f"Predicted vs Actual – {best_name}")

# Distribuția reziduurilor
residuals = y_test - y_pred_best
sns.histplot(residuals, bins=40, kde=True, ax=axes[1], color="coral")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Reziduu ($)")
axes[1].set_title(f"Distribuția Reziduurilor – {best_name}")

plt.tight_layout()
plt.savefig("photos/predicted_vs_actual.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"MAE:  {mean_absolute_error(y_test, y_pred_best):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best)):.2f}")
print(f"R²:   {r2_score(y_test, y_pred_best):.4f}")

MAE:  638.47
RMSE: 734.72
R²:   -0.0120


In [17]:
# Feature importance (dacă modelul suportă)
if hasattr(best_pipe["model"], "feature_importances_"):
    ohe_features = list(best_pipe["prep"].named_transformers_["cat_ohe"]
                        .get_feature_names_out(cat_ohe))
    feature_names = num_cols + ohe_features + cat_ord
    importances   = best_pipe["model"].feature_importances_
    
    feat_df = (pd.DataFrame({"Feature": feature_names, "Importance": importances})
               .sort_values("Importance", ascending=False).head(15))
    
    plt.figure(figsize=(9, 5))
    sns.barplot(data=feat_df, x="Importance", y="Feature", palette="viridis")
    plt.title(f"Top 15 Feature Importances – {best_name}")
    plt.tight_layout()
    plt.savefig("photos/feature_importance.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print(f"{best_name} nu oferă feature importances.")

Ridge Regression nu oferă feature importances.


## 9. Salvare model

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(best_pipe, "models/laptop_price_model.pth")
joblib.dump({
    "cat_ohe": cat_ohe,
    "cat_ord": cat_ord,
    "num_cols": num_cols,
    "target": target,
    "model_name": best_name,
    "metrics": results[best_name],
}, "models/laptop_price_metadata.pth")

print(f"Model salvat: models/laptop_price_model.pth")
print(f"Metadate:     models/laptop_price_metadata.pth")

## 10. Demo predicție

In [19]:
sample = pd.DataFrame([{
    "Brand": "Dell",
    "Processor": "Intel i7",
    "RAM (GB)": 16,
    "Storage (GB)": 512,
    "Screen Size (inches)": 15.6,
    "Graphics Card": "NVIDIA RTX 3060",
    "Operating System": "Windows 11",
    "Weight (kg)": 1.8,
    "Battery Life (hours)": 8.0,
    "Warranty (years)": 2,
}])

pred = best_pipe.predict(sample)[0]
print(f"Dell i7 / 16GB RAM / 512GB SSD / RTX 3060 / Win11")
print(f"Preț estimat: ${pred:.2f}")


Dell i7 / 16GB RAM / 512GB SSD / RTX 3060 / Win11
Preț estimat: $1788.98
